In [ ]:
# ==========================
# 1. Import Libraries
# ==========================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

# New section

In [ ]:
# ==========================
# 2. Load Dataset
# ==========================
df = pd.read_excel("/content/Telco_customer_churn.xlsx")

print(df.head())
print(df.shape)

   CustomerID  Count        Country       State         City  Zip Code  \
0  3668-QPYBK      1  United States  California  Los Angeles     90003   
1  9237-HQITU      1  United States  California  Los Angeles     90005   
2  9305-CDSKC      1  United States  California  Los Angeles     90006   
3  7892-POOKP      1  United States  California  Los Angeles     90010   
4  0280-XJGEX      1  United States  California  Los Angeles     90015   

                 Lat Long   Latitude   Longitude  Gender  ...        Contract  \
0  33.964131, -118.272783  33.964131 -118.272783    Male  ...  Month-to-month   
1   34.059281, -118.30742  34.059281 -118.307420  Female  ...  Month-to-month   
2  34.048013, -118.293953  34.048013 -118.293953  Female  ...  Month-to-month   
3  34.062125, -118.315709  34.062125 -118.315709  Female  ...  Month-to-month   
4  34.039224, -118.266293  34.039224 -118.266293    Male  ...  Month-to-month   

  Paperless Billing             Payment Method  Monthly Charges Tota

In [ ]:
# ==========================
# 3. Data Cleaning
# ==========================

# Remove unnecessary columns
drop_cols = ["CustomerID", "Count"]
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

# Fill missing values
df.fillna(df.mode().iloc[0], inplace=True)

/tmp/ipykernel_476/1586120786.py:10: UserWarning: Unable to sort modes: '<' not supported between instances of 'str' and 'float'
  df.fillna(df.mode().iloc[0], inplace=True)


In [ ]:
import numpy as np

# ==========================
# 4. Encode Categorical Data
# ==========================
# Convert 'Total Charges' to numeric, handling potential non-numeric entries
# Common issue: 'Total Charges' might contain empty strings or spaces for new customers.
# We will coerce errors, turning non-numeric values into NaN.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# Fill NaN values created by coercion (e.g., empty strings became NaN).
# For 'Total Charges', NaN typically means the customer is new and has no charges yet, so fill with 0.
# Updated to avoid FutureWarning: assign the result back to the column instead of using inplace=True.
df['Total Charges'] = df['Total Charges'].fillna(0)

le = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

In [ ]:
# ==========================
# 5. Split Features & Target
# ==========================
X = df.drop("Churn Label", axis=1)
y = df["Churn Label"]

In [ ]:
# ==========================
# 6. Train Test Split
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# ==========================
# 7. Feature Scaling
# ==========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# ==========================
# 8. Train Model
# ==========================
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
# ==========================
# 9. Prediction
# ==========================
y_pred = model.predict(X_test)

In [ ]:
# ==========================
# 10. Evaluation
# ==========================
print("Accuracy :", accuracy_score(y_test, y_pred))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

Accuracy : 1.0

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1035
           1       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409


Confusion Matrix

[[1035    0]
 [   0  374]]


In [ ]:
# ==========================
# 11. Feature Importance
# ==========================
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

importance = importance.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False
)

print(importance.head(10))

              Feature  Coefficient
26        Churn Value     5.045169
27        Churn Score     1.141287
29       Churn Reason     1.097106
24    Monthly Charges     0.279828
21           Contract    -0.265066
11      Tenure Months    -0.261417
10         Dependents    -0.235415
16      Online Backup    -0.095935
22  Paperless Billing     0.086930
18       Tech Support    -0.086547


In [ ]:
# ==========================
# 12. Predict New Customer
# ==========================
sample = X.iloc[[0]]

sample = scaler.transform(sample)

prediction = model.predict(sample)

if prediction[0] == 1:
    print("Customer Will Churn")
else:
    print("Customer Will Not Churn")

Customer Will Churn


In [ ]:
# ==========================
# 13. Save Model
# ==========================
joblib.dump(model, "customer_churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model Saved Successfully")

Model Saved Successfully
